In [2]:
import pandas as pd

# 定义通用大文件读取函数
def read_big_csv(file_path, chunk_size=10000):
    chunk_list = []
    # 分块读取+容错编码
    for chunk in pd.read_csv(file_path, chunksize=chunk_size,
                             encoding="utf-8-sig", encoding_errors="ignore"):
        chunk_list.append(chunk)
    df = pd.concat(chunk_list, ignore_index=True)
    return df

# 读取四份原始数据
df_users = read_big_csv("./data/raw/users.csv")
df_products = read_big_csv("./data/raw/products.csv")
df_browses = read_big_csv("./data/raw/browses.csv")
df_orders = read_big_csv("./data/raw/orders.csv")

print("✅ 四份数据集全部读取完成！")

✅ 四份数据集全部读取完成！


In [4]:
def clean_log(origin_df, clean_df, name):
    print(f"===== {name}清洗日志 =====")
    print(f"清洗前数据量：{origin_df.shape[0]} 行")
    print(f"清洗后数据量：{clean_df.shape[0]} 行")
    print(f"去除无效数据：{origin_df.shape[0] - clean_df.shape[0]} 行\n")

# 备份原始数据
df_users_origin = df_users.copy()
df_products_origin = df_products.copy()
df_browses_origin = df_browses.copy()
df_orders_origin = df_orders.copy()

In [5]:
# 1.主键去重
df_users = df_users.drop_duplicates(subset=["id"], keep="first")
# 2.过滤异常年龄（0-100岁为合理区间）
if "age" in df_users.columns:
    df_users = df_users[(df_users["age"] >= 0) & (df_users["age"] <= 100)]
# 输出清洗日志
clean_log(df_users_origin, df_users, "用户表")

===== 用户表清洗日志 =====
清洗前数据量：100000 行
清洗后数据量：100000 行
去除无效数据：0 行



In [6]:
# 1.主键去重
df_products = df_products.drop_duplicates(subset=["id"], keep="first")
# 2.过滤负价格、负销量异常数据
if "price" in df_products.columns and "sales" in df_products.columns:
    df_products = df_products[(df_products["price"] > 0) & (df_products["sales"] >= 0)]
clean_log(df_products_origin, df_products, "商品表")

===== 商品表清洗日志 =====
清洗前数据量：50000 行
清洗后数据量：48223 行
去除无效数据：1777 行



In [10]:
# 1.核心字段去空
key_cols = ["user_id", "product_id", "created_at"]
df_browses = df_browses.dropna(subset=key_cols)
# 2.去除重复浏览记录
df_browses = df_browses.drop_duplicates(keep="first")
clean_log(df_browses_origin, df_browses, "浏览表")

===== 浏览表清洗日志 =====
清洗前数据量：5000000 行
清洗后数据量：5000000 行
去除无效数据：0 行



In [20]:
df_orders_origin = df_orders.copy()     # 原始数据的副本
# ... 清洗代码 ...
clean_log(df_orders_origin, df_orders, "订单表")

===== 订单表清洗日志 =====
清洗前数据量：1000000 行
清洗后数据量：1000000 行
去除无效数据：0 行



In [19]:
# 批量保存清洗数据
df_users.to_csv("./data/clean/clean_users.csv", index=False, encoding="utf-8-sig")
df_products.to_csv("./data/clean/clean_products.csv", index=False, encoding="utf-8-sig")
df_browses.to_csv("./data/clean/clean_browses.csv", index=False, encoding="utf-8-sig")
df_orders.to_csv("./data/clean/clean_orders.csv", index=False, encoding="utf-8-sig")

print("✅ 所有数据表清洗完成，已保存至data/clean目录！")

✅ 所有数据表清洗完成，已保存至data/clean目录！


In [30]:
df_products.head(10)


,id,name,category_id,category_path,price,original_price,stock,sales,views,favorites,tags,status
0,1,机械革命 移动硬盘,25,2-25,3099.00,3681.35,671,250,6712,142,"正品保障,进口,防水,热销",1
1,2,vivo 智能手表,13,1-13,2299.00,2573.74,2306,241,4756,265,次日达,1
2,3,安踏 自行车,85,8-85,5000.00,6911.49,1882,387,10690,1110,"静音,进口,便携,热销",1
3,4,九牧 餐桌,72,7-72,50.00,58.12,61,232,8639,411,"七天无理由,防水,静音,高清",1
4,5,红米 手机壳,15,1-15,1699.00,2085.72,671,286,14088,285,七天无理由,1
5,6,小米 电饭煲,31,3-31,1899.00,2361.49,637,133,2112,171,正品保障,1
6,7,悦诗风吟 面霜,51,5-51,69.07,97.64,118,299,4291,310,便携,1
7,8,太平鸟 休闲鞋,43,4-43,59.35,86.88,655,114,1853,101,"防水,包邮",1
8,9,红米 充电宝,15,1-15,764.62,1118.41,206,200,8128,508,"防水,便携,限时特惠,新品",1
10,11,vivo 数据线,14,1-14,499.00,567.44,1958,151,6079,302,"环保,品牌直营",1
